[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shreyashgol/phonetic-guided-speech-generation/blob/main/notebooks/15_demo_and_summary.ipynb)

# 15 · Text-to-speech demo and Phase 2 summary
**Goal:** type any English sentence, hear it, and collect the Phase 2 numbers for the report.

**Use a GPU:** Runtime -> Change runtime type -> T4.

**How the demo works**
1. `g2p_en` converts text to ARPAbet phonemes with stress.
2. Silence tokens are added at the start and end and after commas/full stops, with the **median lengths measured on LJSpeech** (`sil_stats.json`).
3. The acoustic model's **own duration head** predicts speech-phone durations. *(The Phase 1 BiLSTM is evaluated on test utterances in notebook 14. Using it on free text needs the Phase 1 feature builder exposed as a function; this is listed as future work.)*
4. Acoustic model → mel → HiFi-GAN → wav.

**Reads:** checkpoint, `sil_stats.json`, `14_*.csv`. **Writes:** `results/audio/15_*.wav`, `results/15_phase2_summary.md`.

In [ ]:
# ---- Cell 0: Setup (identical at the top of every notebook)
import os, sys, json, glob
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/phonetic-tts'
    IN_COLAB = True
except ImportError:                      # running somewhere else (local / tests)
    PROJECT_ROOT = os.environ.get('PROJECT_ROOT', os.getcwd())
    IN_COLAB = False
sys.path.insert(0, PROJECT_ROOT)
from src.utils.config import load_config
cfg = load_config(PROJECT_ROOT)
A = cfg['audio']; P = cfg['abs_paths']
RAW, PROCESSED, ALIGN, SPLITS, RESULTS = P['raw'], P['processed'], P['alignments'], P['splits'], P['results']
for d in (PROCESSED, ALIGN, SPLITS, RESULTS):
    os.makedirs(d, exist_ok=True)
print('Project root:', PROJECT_ROOT, '| Colab:', IN_COLAB)

Mounted at /content/drive
Project root: /content/drive/MyDrive/phonetic-tts | Colab: True


In [ ]:
%%bash
pip -q install speechbrain g2p_en

In [ ]:
# ---- Write shared module: pretrained HiFi-GAN wrapper
import os, importlib
_src = r'''"""Pretrained HiFi-GAN (SpeechBrain, trained on LJSpeech at 22.05 kHz, 80 mels, hop 256)."""
import torch


class Vocoder:
    def __init__(self, device='cpu', savedir='pretrained_hifigan'):
        try:
            from speechbrain.inference.vocoders import HIFIGAN
        except ImportError:                                   # older speechbrain
            from speechbrain.pretrained import HIFIGAN
        self.device = device
        self.m = HIFIGAN.from_hparams(source='speechbrain/tts-hifigan-ljspeech', savedir=savedir, run_opts={'device': device})

    @torch.no_grad()
    def __call__(self, mel):
        """mel: (B, T, 80) or (T, 80) log-mel in natural units (NOT normalised) -> waveform (B, T*256) float tensor on CPU."""
        if mel.dim() == 2: mel = mel.unsqueeze(0)
        w = self.m.decode_batch(mel.transpose(1, 2).float().to(self.device))
        return w.reshape(w.shape[0], -1).cpu()
'''
_p = os.path.join(PROJECT_ROOT, "src/vocoder/hifigan.py")
os.makedirs(os.path.dirname(_p), exist_ok=True)
for _d in ["src", os.path.dirname("src/vocoder/hifigan.py")]:
    open(os.path.join(PROJECT_ROOT, _d, "__init__.py"), "a").close()
open(_p, "w").write(_src)
importlib.invalidate_caches()
print("wrote", _p)

wrote /content/drive/MyDrive/phonetic-tts/src/vocoder/hifigan.py


In [ ]:
# ---- Write shared module: dataset
import os, importlib
_src = r'''"""Phase 2 data loading: manifest + mels from notebook 10."""
import json, os
import numpy as np, pandas as pd, torch
from torch.utils.data import Dataset, Sampler


class TTSDataset(Dataset):
    def __init__(self, processed_dir, split, max_items=None):
        df = pd.read_csv(os.path.join(processed_dir, 'phase2_manifest.csv'))
        df = df[df.split == split].reset_index(drop=True)
        if max_items: df = df.iloc[:max_items]
        st = json.load(open(os.path.join(processed_dir, 'mel_stats.json')))
        self.mean = np.array(st['mean'], np.float32); self.std = np.array(st['std'], np.float32) + 1e-5
        self.mel_dir = os.path.join(processed_dir, 'mels')
        self.ids = df.utt_id.tolist(); self.n_frames = df.n_frames.to_numpy()
        self.tokens = [np.array(s.split(), np.int64) for s in df.tokens]
        self.stress = [np.array(s.split(), np.int64) for s in df.stress]
        self.durs = [np.array(s.split(), np.int64) for s in df.durations]

    def __len__(self):
        return len(self.ids)

    def mel(self, i):
        m = np.load(os.path.join(self.mel_dir, self.ids[i] + '.npy')).astype(np.float32)
        return (m - self.mean) / self.std

    def __getitem__(self, i):
        return dict(id=self.ids[i], tok=torch.from_numpy(self.tokens[i]), stress=torch.from_numpy(self.stress[i]),
                    dur=torch.from_numpy(self.durs[i]), mel=torch.from_numpy(self.mel(i)))


def collate(batch):
    B = len(batch); T = max(len(b['tok']) for b in batch); M = max(len(b['mel']) for b in batch)
    n_mels = batch[0]['mel'].shape[1]
    tok = torch.zeros(B, T, dtype=torch.long); st = torch.zeros(B, T, dtype=torch.long)
    dur = torch.zeros(B, T, dtype=torch.long); mel = torch.zeros(B, M, n_mels)
    for i, b in enumerate(batch):
        t, m = len(b['tok']), len(b['mel'])
        tok[i, :t] = b['tok']; st[i, :t] = b['stress']; dur[i, :t] = b['dur']; mel[i, :m] = b['mel']
    return dict(ids=[b['id'] for b in batch], tok=tok, stress=st, dur=dur, mel=mel,
                mel_len=torch.tensor([len(b['mel']) for b in batch]))


class BucketSampler(Sampler):
    """Batches of similar length (less padding), shuffled each epoch."""
    def __init__(self, n_frames, batch_size, shuffle=True, seed=0):
        self.n = np.asarray(n_frames); self.bs = batch_size; self.shuffle = shuffle; self.epoch = 0; self.seed = seed

    def __iter__(self):
        rng = np.random.RandomState(self.seed + self.epoch); self.epoch += 1
        idx = rng.permutation(len(self.n)) if self.shuffle else np.arange(len(self.n))
        chunk = self.bs * 50
        batches = []
        for s in range(0, len(idx), chunk):
            c = idx[s:s + chunk]; c = c[np.argsort(self.n[c])]
            batches += [c[j:j + self.bs].tolist() for j in range(0, len(c), self.bs)]
        if self.shuffle: rng.shuffle(batches)
        return iter(batches)

    def __len__(self):
        return (len(self.n) + self.bs - 1) // self.bs
'''
_p = os.path.join(PROJECT_ROOT, "src/data/tts_dataset.py")
os.makedirs(os.path.dirname(_p), exist_ok=True)
for _d in ["src", os.path.dirname("src/data/tts_dataset.py")]:
    open(os.path.join(PROJECT_ROOT, _d, "__init__.py"), "a").close()
open(_p, "w").write(_src)
importlib.invalidate_caches()
print("wrote", _p)

wrote /content/drive/MyDrive/phonetic-tts/src/data/tts_dataset.py


In [ ]:
# ---- Write shared module: acoustic model
import os, importlib
_src = r'''"""FastSpeech 2-style acoustic model (phonemes + durations -> mel). No pitch/energy predictors (added later as an ablation)."""
import math
import torch, torch.nn as nn, torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence


def sinusoid(n, d):
    pos = torch.arange(n).unsqueeze(1).float()
    ang = pos / torch.pow(10000.0, torch.arange(0, d, 2).float() / d)
    pe = torch.zeros(n, d); pe[:, 0::2] = torch.sin(ang); pe[:, 1::2] = torch.cos(ang)
    return pe


class FFTBlock(nn.Module):
    """Self-attention + 1D-conv feed-forward, post-LayerNorm (as in FastSpeech)."""
    def __init__(self, d, heads, ffn, k, p):
        super().__init__()
        self.attn = nn.MultiheadAttention(d, heads, dropout=p, batch_first=True)
        self.n1, self.n2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.c1 = nn.Conv1d(d, ffn, k, padding=k // 2); self.c2 = nn.Conv1d(ffn, d, 1)
        self.drop = nn.Dropout(p)

    def forward(self, x, pad):                       # x (B,T,D); pad (B,T) True = padding
        a, _ = self.attn(x, x, x, key_padding_mask=pad, need_weights=False)
        x = self.n1(x + self.drop(a))
        y = x.masked_fill(pad.unsqueeze(-1), 0).transpose(1, 2)
        y = self.c2(F.relu(self.c1(y))).transpose(1, 2)
        x = self.n2(x + self.drop(y))
        return x.masked_fill(pad.unsqueeze(-1), 0)


class DurationPredictor(nn.Module):
    """Predicts log(frames + 1) per token (the model's own duration head)."""
    def __init__(self, d, k=3, p=0.2):
        super().__init__()
        self.c1 = nn.Conv1d(d, d, k, padding=k // 2); self.n1 = nn.LayerNorm(d)
        self.c2 = nn.Conv1d(d, d, k, padding=k // 2); self.n2 = nn.LayerNorm(d)
        self.out = nn.Linear(d, 1); self.drop = nn.Dropout(p)

    def forward(self, x, pad):
        m = pad.unsqueeze(-1)
        y = self.drop(self.n1(F.relu(self.c1(x.masked_fill(m, 0).transpose(1, 2))).transpose(1, 2)))
        y = self.drop(self.n2(F.relu(self.c2(y.masked_fill(m, 0).transpose(1, 2))).transpose(1, 2)))
        return self.out(y).squeeze(-1).masked_fill(pad, 0)


def length_regulate(x, dur):
    """Repeat each token vector dur[b, t] times. x (B,T,D), dur (B,T) long -> (B,M,D), lengths (B,)"""
    outs = [torch.repeat_interleave(x[b], dur[b], dim=0) for b in range(x.size(0))]
    lens = torch.tensor([o.size(0) for o in outs], device=x.device)
    return pad_sequence(outs, batch_first=True), lens


class AcousticModel(nn.Module):
    def __init__(self, n_vocab, n_mels=80, d=256, heads=2, ffn=768, k=3, n_enc=4, n_dec=4, p=0.1, max_len=1600):
        super().__init__()
        self.tok = nn.Embedding(n_vocab, d, padding_idx=0); self.stress = nn.Embedding(4, d)
        self.register_buffer('pe', sinusoid(max_len, d), persistent=False)
        self.enc = nn.ModuleList([FFTBlock(d, heads, ffn, k, p) for _ in range(n_enc)])
        self.dec = nn.ModuleList([FFTBlock(d, heads, ffn, k, p) for _ in range(n_dec)])
        self.dp = DurationPredictor(d); self.out = nn.Linear(d, n_mels)

    def forward(self, tok, stress, dur=None, alpha=1.0):
        """dur given -> teacher forcing (training / ground truth / Phase 1 durations). dur None -> own duration head."""
        pad = tok == 0
        x = self.tok(tok) + self.stress(stress) + self.pe[:tok.size(1)]
        for b in self.enc: x = b(x, pad)
        logd = self.dp(x, pad)
        if dur is None:
            dur = torch.clamp(torch.round((torch.exp(logd) - 1) * alpha), min=1).long().masked_fill(pad, 0)
        y, lens = length_regulate(x, dur)
        mpad = torch.arange(y.size(1), device=y.device)[None] >= lens[:, None]
        y = (y + self.pe[:y.size(1)]).masked_fill(mpad.unsqueeze(-1), 0)
        for b in self.dec: y = b(y, mpad)
        return dict(mel=self.out(y), logd=logd, tok_pad=pad, mel_pad=mpad, dur=dur)


def acoustic_loss(o, mel, dur):
    """masked L1 on mel + masked MSE on log-duration."""
    assert o['mel'].shape == mel.shape, (o['mel'].shape, mel.shape)
    m = (~o['mel_pad']).unsqueeze(-1).float()
    mel_l = ((o['mel'] - mel).abs() * m).sum() / (m.sum() * mel.size(-1))
    t = (~o['tok_pad']).float()
    dur_l = (((o['logd'] - torch.log(dur.float() + 1)) ** 2) * t).sum() / t.sum()
    return mel_l + dur_l, mel_l, dur_l
'''
_p = os.path.join(PROJECT_ROOT, "src/models/acoustic/model.py")
os.makedirs(os.path.dirname(_p), exist_ok=True)
for _d in ["src", os.path.dirname("src/models/acoustic/model.py")]:
    open(os.path.join(PROJECT_ROOT, _d, "__init__.py"), "a").close()
open(_p, "w").write(_src)
importlib.invalidate_caches()
print("wrote", _p)

wrote /content/drive/MyDrive/phonetic-tts/src/models/acoustic/model.py


## Step 1: Load everything

In [ ]:
import re, numpy as np, pandas as pd, torch, soundfile as sf
from IPython.display import Audio, display
from src.models.acoustic.model import AcousticModel
from src.vocoder.hifigan import Vocoder
from g2p_en import G2p
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
contract = json.load(open(os.path.join(PROCESSED, 'phase2_contract.json'))); VOCAB = contract['vocab']; SIL = VOCAB['sil']
sil = json.load(open(os.path.join(PROCESSED, 'sil_stats.json'))); st = json.load(open(os.path.join(PROCESSED, 'mel_stats.json')))
MEAN, STD = np.array(st['mean'], np.float32), np.array(st['std'], np.float32) + 1e-5
ck = torch.load(os.path.join(PROJECT_ROOT, 'checkpoints', 'acoustic', 'best.pt'), map_location=DEV, weights_only=False)
model = AcousticModel(len(VOCAB), A['n_mels'], **ck['arch']).to(DEV); model.load_state_dict(ck['model']); model.eval()
voc = Vocoder(DEV, os.path.join(PROJECT_ROOT, 'pretrained', 'hifigan')); g2p = G2p()
AUDIO = os.path.join(RESULTS, 'audio'); os.makedirs(AUDIO, exist_ok=True)
print('ready | checkpoint step', ck['step'], '| unknown phones are skipped; vocab:', len(VOCAB))

[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package cmudict to /root/nltk_data...
[nltk_data]   Unzipping corpora/cmudict.zip.
INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Fetching from HuggingFace Hub 'speechbrain/tts-hifigan-ljspeech' if not cached


hyperparams.yaml:   0%|          | 0.00/1.16k [00:00<?, ?B/s]

INFO:speechbrain.utils.fetching:Fetch generator.ckpt: Fetching from HuggingFace Hub 'speechbrain/tts-hifigan-ljspeech' if not cached


generator.ckpt: reconstructing file:   0%|          |  0.00B / 55.8MB            

generator.ckpt: downloading bytes:           |  0.00B            

INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: generator


ready | checkpoint step 8000 | unknown phones are skipped; vocab: 41


## Step 2: Text -> tokens

In [ ]:
import nltk
nltk.download('averaged_perceptron_tagger_eng')

def text_to_tokens(text):
    toks, stress = [SIL], [0]; internal = int(round(sil['internal']))
    for p in g2p(text.lower()):
        if p == ' ': continue
        if p in ',;:.!?':
            if toks[-1] != SIL: toks.append(SIL); stress.append(0)
            continue
        m = re.fullmatch(r'([A-Z]+)([012])?', p)
        if not m or m.group(1) not in VOCAB: continue
        toks.append(VOCAB[m.group(1)]); stress.append(0 if m.group(2) is None else int(m.group(2)) + 1)
    if toks[-1] != SIL: toks.append(SIL); stress.append(0)
    return np.array(toks), np.array(stress)

@torch.no_grad()
def speak(text, alpha=1.0, name=None):
    tok, stress = text_to_tokens(text); b = lambda a: torch.from_numpy(a)[None].to(DEV)
    d = model(b(tok), b(stress), alpha=alpha)['dur'][0].cpu().numpy()
    n = len(d); is_sil = tok == SIL; pos = np.arange(n)
    d[is_sil] = int(round(sil['internal']))                                  # internal pauses
    d[is_sil & (pos == 0)] = int(round(sil['leading'])); d[is_sil & (pos == n - 1)] = int(round(sil['trailing']))
    o = model(b(tok), b(stress), dur=torch.from_numpy(d)[None].to(DEV))
    mel = o['mel'][0].cpu().numpy() * STD + MEAN
    w = np.clip(voc(torch.from_numpy(mel.astype(np.float32)))[0].numpy(), -1, 1)
    if name: sf.write(os.path.join(AUDIO, f'15_{name}.wav'), w, A['sample_rate'])
    return w, d

for k, t in enumerate(['Phonetic guided speech generation uses explicit phoneme durations.',
                       'The quick brown fox jumps over the lazy dog, and then it runs away.',
                       'Printing, in the only sense with which we are at present concerned, differs from most if not from all the arts.']):
    w, d = speak(t, name=f'demo{k+1}'); print(t, f'| {len(w)/A["sample_rate"]:.1f}s'); display(Audio(w, rate=A['sample_rate']))

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


Phonetic guided speech generation uses explicit phoneme durations. | 4.7s


The quick brown fox jumps over the lazy dog, and then it runs away. | 4.7s


Printing, in the only sense with which we are at present concerned, differs from most if not from all the arts. | 6.8s


## Step 3: Speed control (the duration is an explicit knob)
`alpha > 1` stretches the durations (slower), `alpha < 1` speeds up. This is something a model without explicit durations cannot do directly, and a good point for the viva.

In [ ]:
t = 'Duration control is a direct benefit of predicting phoneme lengths.'
for a in (0.8, 1.0, 1.25):
    w, d = speak(t, alpha=a, name=f'speed_{a}'); print(f'alpha={a}: {len(w)/A["sample_rate"]:.2f}s'); display(Audio(w, rate=A['sample_rate']))

alpha=0.8: 3.45s


alpha=1.0: 4.23s


alpha=1.25: 5.28s


## Step 4: Collect Phase 2 results for the report

In [ ]:
parts = ['# Phase 2 summary (auto-generated)', '', f'Acoustic model: {sum(p.numel() for p in model.parameters())/1e6:.2f} M parameters, best checkpoint at step {ck["step"]}, best validation mel L1 {ck["best"]:.4f} (normalised units).']
for f, title in [('14_objective.csv', 'Objective duration-swap results (test)'), ('14_paired.csv', 'Paired differences'), ('14_mos.csv', 'Listening test (MOS)'), ('13_copy_synthesis.csv', 'Vocoder copy-synthesis')]:
    p = os.path.join(RESULTS, f)
    if os.path.exists(p): parts += ['', f'## {title}', '', pd.read_csv(p).to_markdown(index=False)]
    else: parts += ['', f'## {title}', '', f'(missing: run the notebook that writes {f})']
open(os.path.join(RESULTS, '15_phase2_summary.md'), 'w').write(chr(10).join(parts)); print(chr(10).join(parts))

# Phase 2 summary (auto-generated)

Acoustic model: 8.84 M parameters, best checkpoint at step 8000, best validation mel L1 0.4073 (normalised units).

## Objective duration-swap results (test)

| condition     |   dtw_l1 |   ci_lo |   ci_hi |   total_len_err_frames |   len_ci_lo |   len_ci_hi |
|:--------------|---------:|--------:|--------:|-----------------------:|------------:|------------:|
| ground_truth  |   0.6617 |  0.6562 |  0.6674 |                  0     |      0      |      0      |
| phase1_bilstm |   0.6646 |  0.6592 |  0.6702 |                 20.15  |     17.5998 |     22.6751 |
| phone_mean    |   0.6782 |  0.6727 |  0.6837 |                 48.45  |     43.575  |     53.4701 |
| global_mean   |   0.6859 |  0.6808 |  0.6914 |                 42.25  |     37.6699 |     47.1006 |
| own_head      |   0.6661 |  0.6606 |  0.6715 |                 19.855 |     17.3249 |     22.435  |

## Paired differences

| comparison                   |   mean_diff |   ci_lo |   ci_hi | 

## Phase 2 checklist for the report and viva
- [ ] Architecture figure updated: the Phase 2 block (dashed orange) now has measured numbers.
- [ ] Report sections: acoustic model design, training (curves from `12_curves.png`), vocoder choice and copy-synthesis check, duration-swap ablation (`14_objective.csv`, `14_paired.csv`), listening test (`14_mos.csv`, number of raters and clips).
- [ ] State clearly: vocoder is **pretrained**; pitch/energy predictors not included; the demo uses the model's own duration head.
- [ ] Limitations: small model, single speaker, DTW mel distance is a proxy, listening test is small.
- [ ] Viva one-liners: *why durations as input* (control, stability, no attention failures), *why teacher forcing* (stable training), *why the DTW metric* (lengths differ), *what the ablation shows* (read from your own table).

**Future work:** pitch/energy predictors (full FastSpeech 2), exposing the Phase 1 feature builder to drive durations from free text, training on more speakers.